## Imports

In [ ]:
!pip install -q -U transformers datasets accelerate peft bitsandbytes

In [ ]:
import torch
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    prepare_model_for_kbit_training,
    PeftModel
)

## Configuration & Model Selection

In [ ]:
SUPPORTED_MODELS = {
    "t5": {
        "id": "t5-base",
        "type": "seq2seq",
        "target_modules": ["q", "v"],
        # "target_modules": ["q", "k", "v", "o", "wi", "wo"],
    },
    "flan-t5": {
        "id": "google/flan-t5-base",
        "type": "seq2seq",
        "target_modules": ["q", "v"],
        # "target_modules": ["q", "k", "v", "o", "wi", "wo"],
    },
    "bart": {
        "id": "facebook/bart-base",
        "type": "seq2seq",
        "target_modules":["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"],
    },
    "gpt2": {
        "id": "gpt2-large",
        "type": "causal",
        "target_modules": ["c_attn"],
        # "target_modules": ["c_attn", "c_proj", "c_fc"],
    }
}

# ---> SWITCH MODEL HERE <---
ACTIVE_MODEL_KEY = "flan-t5"
MODEL_CFG = SUPPORTED_MODELS[ACTIVE_MODEL_KEY]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device} | Active Model: {MODEL_CFG['id']}")

## Data Loading & Unified Schema Preparation

In [ ]:
df1 = pd.read_json("nonsarcastic_to_sarcastic_similarityfiltered.json", lines=True)
df2 = pd.read_json("sarcastic_to_nonsarcastic_filtered_similarity.json", lines=True)

df1["source"] = "ns2s"
df2["source"] = "s2ns"

df = pd.concat([df1, df2], ignore_index=True)

train_df, val_df = train_test_split(
    df,
    test_size=0.15,
    random_state=42,
    shuffle=True,
    stratify=df["source"]
)

def to_unified_schema(row):
    """Maps the raw sarcasm dataset into the unified Alpaca-style schema."""
    return {
        "instruction": "Convert to sarcastic" if row['is_sarcastic'] == 0 else "Convert to normal",
        "input": row['headline'],
        "output": row['reversed_headline']
    }

# Convert to huggingface datasets
hf_datasets = []
for split_df in [train_df, val_df]:
    # Apply the schema formatting
    formatted_data = split_df.apply(to_unified_schema, axis=1).tolist()
    # Convert list of dicts directly into a Hugging Face Dataset
    hf_datasets.append(Dataset.from_list(formatted_data))

train_dataset, val_dataset = hf_datasets

## Prompt Templates & Tokenization

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CFG['id'])
if MODEL_CFG["type"] == "causal" and tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def build_prompt(instruction, input_text, model_type):
    """Constructs the prompt string dynamically based on the model architecture."""
    if model_type == "causal":
        # Verbose formatting works best for Decoder-only models
        prompt = f"### Instruction:\n{instruction}\n\n"
        if input_text:
            prompt += f"### Input:\n{input_text}\n\n"
        prompt += "### Response:\n"
        return prompt
    else:
        # Minimal formatting works best for Encoder-Decoder models (T5/BART)
        prompt = f"{instruction}\n\n"
        if input_text:
            prompt += f"Input: {input_text}"
        return prompt

def preprocess_function(examples):
    # Extract lists of data
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]

    # Build the specific prompts for this batch
    prompts =[
        build_prompt(inst, inp, MODEL_CFG["type"])
        for inst, inp in zip(instructions, inputs)
    ]

    if MODEL_CFG["type"] == "seq2seq":
        # Seq2Seq needs inputs and labels tokenized separately
        model_inputs = tokenizer(prompts, max_length=256, truncation=True)
        labels = tokenizer(text_target=outputs, max_length=128, truncation=True)
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    else:
        # Combine the prompt and the expected output + EOS token
        combined_texts =[
            f"{prompt}{target}{tokenizer.eos_token}" 
            for prompt, target in zip(prompts, outputs)
        ]
        
        # Tokenize the full sequences
        model_inputs = tokenizer(combined_texts, max_length=256, truncation=True)
        
        # Tokenize JUST the prompts so we know how many tokens they take up
        # (We use the exact same tokenizer settings so the token count matches)
        prompt_inputs = tokenizer(prompts, max_length=256, truncation=True)
        
        # Copy input_ids to create our labels
        labels = [list(ids) for ids in model_inputs["input_ids"]]
        
        # Loop through every item in the batch to mask the prompt
        for i in range(len(labels)):
            # Find out how many tokens the prompt used
            prompt_len = len(prompt_inputs["input_ids"][i])
            
            # Safety check: in case truncation cut off part of the prompt
            prompt_len = min(prompt_len, len(labels[i]))
            
            # Replace all prompt tokens in the labels array with -100
            labels[i][:prompt_len] =[-100] * prompt_len
            
        model_inputs["labels"] = labels
        return model_inputs

# Map datasets to tokens and drop the raw string columns
cols_to_remove = ["instruction", "input", "output"]
tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=cols_to_remove)
tokenized_val = val_dataset.map(preprocess_function, batched=True, remove_columns=cols_to_remove)

## Model Initialization & QLoRA Setup

In [ ]:
# 4-bit Quantization Config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load the base model
if MODEL_CFG["type"] == "seq2seq":
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CFG['id'], quantization_config=bnb_config, device_map="auto")
    task_type = TaskType.SEQ_2_SEQ_LM
else:
    model = AutoModelForCausalLM.from_pretrained(MODEL_CFG['id'], quantization_config=bnb_config, device_map="auto")
    task_type = TaskType.CAUSAL_LM

# Prepares the model for parameter-efficient fine-tuning (PEFT)
model = prepare_model_for_kbit_training(model)

# Set lora_r
lora_r = 64

lora_alpha = lora_r / 4

# LoRA Configuration
lora_config = LoraConfig(
    r=lora_r,                                  # Rank: determines adapter size (higher = more capacity but slower)
    lora_alpha=lora_alpha,                         # Scaling factor for the adapter
    target_modules=MODEL_CFG["target_modules"], # Which layers to inject adapters into
    lora_dropout=0.05,
    bias="none",
    task_type=task_type
)

# Wrap the base model with PEFT so only the LoRA adapters are trainable
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Training

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

if MODEL_CFG["type"] == "seq2seq":
    # ---------------------------------------------------------
    # ENCODER-DECODER SETTINGS (T5, BART)
    # ---------------------------------------------------------
    training_args = Seq2SeqTrainingArguments(
        output_dir=f"./{ACTIVE_MODEL_KEY}-headline-generator",
        save_strategy="epoch",
        eval_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        per_device_train_batch_size=32,
        gradient_accumulation_steps=1,
        fp16=False,
        logging_steps=10,
        logging_first_step=True,
        optim='adamw_torch',
        max_grad_norm=0.3,
        warmup_ratio=0.03,
        lr_scheduler_type="linear",

        # Seq2Seq Specific Hyperparameters
        learning_rate=5e-4,               # Higher LR
        num_train_epochs=15,
        save_total_limit=2,
        predict_with_generate=True,
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        data_collator=data_collator,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )

else:
    # ---------------------------------------------------------
    # DECODER-ONLY SETTINGS (GPT-2, LLaMA)
    # ---------------------------------------------------------
    training_args = TrainingArguments(
        output_dir=f"./{ACTIVE_MODEL_KEY}-headline-generator",
        save_strategy="epoch",
        eval_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",

        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        fp16=False,
        logging_steps=10,
        logging_first_step=True,
        optim='adamw_torch',
        max_grad_norm=0.3,
        warmup_ratio=0.03,
        lr_scheduler_type="linear",

        # Causal Specific Hyperparameters
        learning_rate=2e-4,               # Lower LR
        num_train_epochs=15,
        save_total_limit=2,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        data_collator=data_collator,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )

print(f"Starting Training for {ACTIVE_MODEL_KEY}...")
trainer.train()

peft_model_path = f"./{ACTIVE_MODEL_KEY}-adapters"

trainer.save_model(peft_model_path)
tokenizer.save_pretrained(peft_model_path)
print(f"Adapters saved to {peft_model_path}")

## Testing

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(peft_model_path)

# Corrected: Must pass the bnb_config again during inference so the shapes match
if MODEL_CFG["type"] == "seq2seq":
    base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CFG['id'], device_map="auto")
else:
    base_model = AutoModelForCausalLM.from_pretrained(MODEL_CFG['id'], device_map="auto")

# Merge the adapters onto the base model
model = PeftModel.from_pretrained(base_model, peft_model_path, device_map="auto")
model.eval() 

def convert_headline(source_headline, to_sarcastic=True):
    instruction = "Convert to sarcastic" if to_sarcastic else "Convert to normal"
    
    # 1. Build the prompt using the exact same function we used in training
    formatted_prompt = build_prompt(
        instruction=instruction, 
        input_text=source_headline, 
        model_type=MODEL_CFG["type"]
    )
    
    # 2. Tokenize input
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

    # 3. Generate
    with torch.no_grad():
        output_tokens = model.generate(
            **inputs,
            max_new_tokens=64,   
            do_sample=False,     
            num_beams=5,         
            early_stopping=True,
            pad_token_id=tokenizer.pad_token_id
        )

    # 4. Decode
    if MODEL_CFG["type"] == "seq2seq":
        headline = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
    else:
        input_length = inputs.input_ids.shape[1]
        generated_tokens = output_tokens[0][input_length:]
        headline = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
        
    return headline

In [ ]:
# This is not the actual test set

tests_normal = [
    'Global Efforts Reduce Plastic Pollution in Oceans by 15%',
    'Scientists Discover Promising Treatment for Alzheimer’s Disease',
    'Community Garden Project Brings Fresh Produce to City Residents',
    'New Renewable Energy Plant Powers 50,000 Homes',
    'Local School Wins National Robotics Championship'
]

tests_sarcastic = [
    'Breaking: Water Still Wet, Scientists Shocked',
    'New Study Confirms That People Who Sleep Less Feel Tired',
    'Politician Finally Keeps Promise, World Completely Astounded',
    'Local Coffee Shop Runs Out of Coffee, Customers Devastated',
    'Man Surprised His Email Didn’t Send Instantly, Technology Fails Again'
]

print("Converting to Sarcastic:")
for test_normal in tests_normal:
    print("Source:", test_normal)
    print("Result:", convert_headline(test_normal, to_sarcastic=True))

print("-" * 50)

print("Converting to Normal:")
for test_sarcastic in tests_sarcastic:
    print("Source:", test_sarcastic)
    print("Result:", convert_headline(test_sarcastic, to_sarcastic=False))